# dbt-modern-stack — walkthrough

This notebook drives the **pure-pandas core** of the project: the
data-quality runner (`dwh.dq`) that mirrors and extends dbt's generic
tests, and the dimensional helpers (`dwh.dimensional`) including the
SCD Type-2 snapshot. No dbt / duckdb / warehouse needed.


In [ ]:
from datetime import datetime, timedelta

import pandas as pd

from dwh import TestSpec, run_suite, scd2_snapshot, suite_failed
from dwh.demo import run_demo

summary = run_demo(0)
print(summary['num_tests'], 'tests,', summary['num_passed'], 'passed')
print('dirty contrast: ', summary['dirty_num_failed'], 'of',
      summary['dirty_num_tests'], 'fail')


## 1. The end-to-end demo

`run_demo(0)` synthesises a small, seeded IMDb-like dataset, builds
marts with the real helpers, and runs the generic-test suite. The clean
marts pass every test; a planted-defect contrast run fails some.


In [ ]:
titles = pd.DataFrame({
    'title_id': ['tt1', 'tt2', 'tt2'],   # planted duplicate
    'title_type': ['movie', 'short', 'short'],
})
ratings = pd.DataFrame({'title_id': ['tt1', 'tt9']})  # tt9 is an orphan

classic = run_suite([
    TestSpec('not_null', titles, 'title_id', table='stg_titles'),
    TestSpec('unique', titles, 'title_id', table='stg_titles'),
    TestSpec('accepted_values', titles, 'title_type', table='stg_titles',
             values=['movie', 'short']),
    TestSpec('relationships', ratings, 'title_id', table='fct',
             parent=titles, parent_col='title_id'),
])
classic


## 2. The four classic generic tests

`not_null`, `unique`, `accepted_values`, `relationships` — each returns
`(passed, failures)` with dbt's exact semantics (e.g. `unique` ignores
NULLs).


In [ ]:
df = pd.DataFrame({
    'rating': [5.0, 0.0, 11.2],            # 0.0 and 11.2 out of [1, 10]
    'start_year': [2000, 2010, 1999],
    'end_year': [2005, 2009, 1999],        # row 1 violates end >= start
    'status': ['released', 'released', 'upcoming'],
    'votes': [10, None, None],             # released row 1 missing votes
    '_loaded_at': ['2026-01-08', '2026-01-08', '2026-01-08'],
})
now = datetime(2026, 1, 10, 12, 0)
extended = run_suite([
    TestSpec('accepted_range', df, 'rating', lo=1.0, hi=10.0),
    TestSpec('expression', df, column='end_year',
             expr_fn=lambda d: d['end_year'] >= d['start_year']),
    TestSpec('not_null_where', df, 'votes',
             predicate=lambda d: d['status'] == 'released'),
    TestSpec('freshness', df, ts_col='_loaded_at',
             max_age=timedelta(hours=24), now=now),
])
extended


## 3. Extended test types

Beyond the four built-ins: `accepted_range`, `not_null_where`,
`expression` (a row-level boolean rule), and relational `freshness`.
Each is a hand-checkable known answer on a tiny frame.


In [ ]:
nulls = pd.DataFrame({'x': [None, 1.0]})
sev = run_suite([
    TestSpec('not_null', nulls, 'x', table='t', severity='warn'),
    TestSpec('unique', nulls, 'x', table='t', severity='error'),
])
print(sev[['test', 'severity', 'passed', 'failures']])
print('suite_failed (warn does not count):', suite_failed(sev))


## 4. Severity levels

Every `TestSpec` runs at `error` (default) or `warn`. A failing `warn`
shows on its row but does not fail the suite — `suite_failed` only looks
at `error` tests, exactly like dbt's `severity: warn`.


In [ ]:
run1 = scd2_snapshot(
    pd.DataFrame({'title_id': ['tt1', 'tt2'],
                  'title_type': ['movie', 'short']}),
    None, key='title_id', cols=['title_type'], valid_from='2024-01-01')

run2 = scd2_snapshot(
    pd.DataFrame({'title_id': ['tt1', 'tt2', 'tt3'],
                  'title_type': ['tvSeries', 'short', 'documentary']}),
    run1, key='title_id', cols=['title_type'], valid_from='2024-06-01')
run2


## 5. SCD Type-2 snapshot

`scd2_snapshot` advances versioned history by one run: unchanged keys
carry forward, a changed key has its prior version closed
(`valid_to` set, `is_current=False`) and a new current version appended,
and new keys are inserted — the history `dbt snapshot` maintains.
